# Birds classification with PointNet

Ноутбук для Google Colab: монтирование Google Drive, клонирование репозитория, установка зависимостей, обучение `birds` на `PointNet`, возобновление обучения и инференс на одном `.txt` файле.

In [ ]:
from pathlib import Path
import shlex
import subprocess

REPO_URL = "https://github.com/djachenkoroman2/magicnet-cloud.git"
PROJECT_ROOT = Path("/content")
REPO_DIR = PROJECT_ROOT / "magicnet-cloud"

BIRDS_DATA_DIR = Path("/content/drive/MyDrive/data/birds")
LOG_ROOT = Path("/content/drive/MyDrive/logs")
CONFIG_PATH = "cfgs/birds/pointnet.yaml"

TRAIN_EPOCHS = 50
RESUME_EPOCHS = 100
TRAIN_BATCH_SIZE = 32
VAL_BATCH_SIZE = 64
NUM_WORKERS = 2
TOPK = 5

# Укажите путь вручную, если хотите проверить конкретный файл.
INFER_FILE = None


def run_cmd(cmd, env=None, cwd=None):
    cwd = REPO_DIR if cwd is None else Path(cwd)
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(printable)
    subprocess.run([str(part) for part in cmd], check=True, env=env, cwd=str(cwd))


def find_latest_checkpoint(log_root=LOG_ROOT, task_name="birds", kind="latest"):
    task_dir = Path(log_root) / task_name
    if not task_dir.exists():
        raise FileNotFoundError(f"Каталог логов не найден: {task_dir}")

    run_dirs = sorted(
        [path for path in task_dir.iterdir() if path.is_dir()],
        key=lambda path: path.stat().st_mtime,
    )
    if not run_dirs:
        raise FileNotFoundError(f"В {task_dir} не найдено ни одного run-каталога")

    run_dir = run_dirs[-1]
    ckpt_dir = run_dir / "checkpoint"
    suffix = "_ckpt_latest.pth" if kind == "latest" else "_ckpt_best.pth"
    matches = sorted(ckpt_dir.glob(f"*{suffix}"), key=lambda path: path.stat().st_mtime)
    if not matches:
        raise FileNotFoundError(f"В {ckpt_dir} не найден checkpoint вида *{suffix}")

    return run_dir, matches[-1]


print(f"REPO_DIR      : {REPO_DIR}")
print(f"BIRDS_DATA_DIR: {BIRDS_DATA_DIR}")
print(f"LOG_ROOT      : {LOG_ROOT}")
print(f"CONFIG_PATH   : {CONFIG_PATH}")

## 1. Монтирование Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Клонирование репозитория

In [ ]:
import os

if not REPO_DIR.exists():
    run_cmd(["git", "clone", REPO_URL, REPO_DIR], cwd=PROJECT_ROOT)
else:
    print(f"Репозиторий уже существует: {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Текущая рабочая директория: {Path.cwd()}")

## 3. Установка зависимостей `install_colab_requirements.sh`

`PointNet` для `birds` не требует дополнительных C++/CUDA расширений, поэтому здесь достаточно базовой установки.

In [ ]:
import os

env = os.environ.copy()
env.setdefault("INSTALL_TORCH_SCATTER", "0")
env.setdefault("INSTALL_POINTNET2_BATCH", "0")
env.setdefault("INSTALL_POINTOPS", "0")
env.setdefault("INSTALL_CHAMFER_DIST", "0")
env.setdefault("INSTALL_EMD", "0")
env.setdefault("INSTALL_SUBSAMPLING", "0")

run_cmd(["bash", "script/install_colab_requirements.sh"], env=env)

## 4. Основное обучение на модели PointNet

Ожидается, что датасет лежит в `BIRDS_DATA_DIR`, например в `/content/drive/MyDrive/data/birds`.

In [ ]:
if not BIRDS_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Датасет birds не найден: {BIRDS_DATA_DIR}. Проверьте путь в первой ячейке."
    )

LOG_ROOT.mkdir(parents=True, exist_ok=True)

train_cmd = [
    "bash",
    "script/main_classification.sh",
    CONFIG_PATH,
    "--data",
    str(BIRDS_DATA_DIR),
    "--log",
    str(LOG_ROOT),
    f"epochs={TRAIN_EPOCHS}",
    f"batch_size={TRAIN_BATCH_SIZE}",
    f"val_batch_size={VAL_BATCH_SIZE}",
    f"dataloader.num_workers={NUM_WORKERS}",
]

run_cmd(train_cmd)

## 5. Возобновление обучения

Ячейка автоматически ищет последний `*_ckpt_latest.pth` в `logs/birds/...`. Значение `RESUME_EPOCHS` должно быть больше эпохи, на которой был сохранён чекпоинт.

In [ ]:
run_dir, resume_ckpt = find_latest_checkpoint(kind="latest")
print(f"Последний run: {run_dir}")
print(f"Checkpoint для resume: {resume_ckpt}")

resume_cmd = [
    "bash",
    "script/main_classification.sh",
    CONFIG_PATH,
    "--data",
    str(BIRDS_DATA_DIR),
    "--log",
    str(LOG_ROOT),
    "--resume",
    str(resume_ckpt),
    f"epochs={RESUME_EPOCHS}",
    f"batch_size={TRAIN_BATCH_SIZE}",
    f"val_batch_size={VAL_BATCH_SIZE}",
    f"dataloader.num_workers={NUM_WORKERS}",
]

run_cmd(resume_cmd)

## 6. Инференс на одном файле

Если `INFER_FILE = None`, ячейка возьмёт первый доступный `.txt` файл из `BIRDS_DATA_DIR`. Для инференса используется последний `*_ckpt_best.pth`.

In [ ]:
import sys
import numpy as np
import torch

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from openpoints.models import build_model_from_cfg
from openpoints.utils import EasyConfig, load_checkpoint


def load_birds_xyz_array(file_path):
    points = np.loadtxt(file_path, dtype=np.float32)
    if points.ndim == 1:
        points = points[None, :]
    if points.shape[1] < 3:
        raise ValueError(
            f"Файл {file_path} должен содержать как минимум три колонки `x y z`, получено shape={points.shape}"
        )
    return np.asarray(points[:, :3], dtype=np.float32)


def normalize_points(points):
    centroid = points.mean(axis=0, keepdims=True)
    points = points - centroid
    scale = np.linalg.norm(points, axis=1).max()
    if scale > 0:
        points = points / scale
    return points.astype(np.float32)


def resample_eval_points(points, num_points):
    num_available = points.shape[0]
    if num_available == 0:
        raise ValueError("Облако точек не может быть пустым")
    if num_available >= num_points:
        idx = np.linspace(0, num_available - 1, num_points, dtype=np.int64)
    else:
        extra = np.arange(num_points - num_available, dtype=np.int64) % num_available
        idx = np.concatenate([np.arange(num_available, dtype=np.int64), extra])
    return np.asarray(points[idx], dtype=np.float32)


if not BIRDS_DATA_DIR.exists():
    raise FileNotFoundError(f"Датасет birds не найден: {BIRDS_DATA_DIR}")

class_names = sorted(
    path.name for path in BIRDS_DATA_DIR.iterdir()
    if path.is_dir() and path.name != "splits"
)
if not class_names:
    raise FileNotFoundError(f"В {BIRDS_DATA_DIR} не найдены папки классов")

cfg = EasyConfig()
cfg.load(str(REPO_DIR / CONFIG_PATH), recursive=True)
cfg.update([f"dataset.common.data_root={BIRDS_DATA_DIR}"])
cfg.resolve_references()

num_points = int(cfg.get("num_points", cfg.dataset.common.get("num_points", 1024)))
normalize = bool(cfg.dataset.common.get("normalize", False))
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

_, best_ckpt = find_latest_checkpoint(kind="best")

if INFER_FILE is None:
    sample_files = sorted(
        path for path in BIRDS_DATA_DIR.rglob("*.txt")
        if "splits" not in path.parts
    )
    if not sample_files:
        raise FileNotFoundError(f"В {BIRDS_DATA_DIR} не найдено ни одного .txt файла для инференса")
    infer_path = sample_files[0]
else:
    infer_path = Path(INFER_FILE)
    if not infer_path.is_file():
        raise FileNotFoundError(f"Файл для инференса не найден: {infer_path}")

points = load_birds_xyz_array(infer_path)
points = resample_eval_points(points, num_points)
if normalize:
    points = normalize_points(points)

pos = torch.from_numpy(points).float().unsqueeze(0).to(device)
batch = {
    "pos": pos.contiguous(),
    "x": pos.transpose(1, 2).contiguous(),
}

model = build_model_from_cfg(cfg.model).to(device)
load_checkpoint(model, str(best_ckpt))
model.eval()

with torch.no_grad():
    logits = model(batch)
    probs = torch.softmax(logits, dim=1)[0].cpu()

if len(class_names) != probs.numel():
    raise ValueError(
        f"Число классов в данных ({len(class_names)}) не совпадает с выходом модели ({probs.numel()})"
    )

topk = min(TOPK, probs.numel())
scores, indices = torch.topk(probs, k=topk)
predicted_class = class_names[int(indices[0])]

print(f"Checkpoint: {best_ckpt}")
print(f"Файл: {infer_path}")
if infer_path.parent.name != "splits":
    print(f"Класс по имени родительской папки: {infer_path.parent.name}")
print(f"Предсказанный класс: {predicted_class}")
print("\nTop predictions:")
for rank, (score, index) in enumerate(zip(scores.tolist(), indices.tolist()), start=1):
    print(f"{rank}. {class_names[int(index)]}: {score:.4f}")